# L16 demo: structured extraction from datasheets

We turn messy component datasheets into a validated parts table, and show what keeps it
honest: a schema, a validator, a repair loop, cost accounting, and a gold set.

> Companion notes: [`notes.md`](notes.md). Assignment: **A8** builds this for real.

**Runs for everyone.** With no API key this notebook uses a deterministic stand-in that
returns provider-shaped responses, so every step executes offline. Set `ANTHROPIC_API_KEY`
(or `OPENAI_API_KEY`) and the same call site hits a real model instead.

## Setup

In [ ]:
import os, json, re, time
from pydantic import BaseModel, field_validator, ValidationError

HAVE_KEY = bool(os.environ.get('ANTHROPIC_API_KEY') or os.environ.get('OPENAI_API_KEY'))
print('live API' if HAVE_KEY else 'offline stand-in (no key found): the demo still runs end to end')

# Claude Sonnet 5 pricing, observed 2026-08-18. Providers change this; pin what you use.
PRICE_IN_PER_MTOK = 2.0
PRICE_OUT_PER_MTOK = 10.0
MODEL = 'claude-sonnet-5'   # pin the exact model id in your report

## The datasheets

Four heterogeneous parts, no two laid out the same. Note the traps: the valve quotes
pressure in **bar**, not MPa, and the bolt has **no pressure rating at all** (a good
extractor must return null, not invent one).

In [ ]:
DATASHEETS = {
'FS-BV200-SS': '''FLOWSTAR BV-200 SERIES | 2-INCH STAINLESS BALL VALVE
Part No.: FS-BV200-SS
Body material: SS316L stainless steel
Working pressure: 40 bar max at 20 C
Temperature range: -20 C to 180 C
Mass: 3.4 kg | End connection: ANSI 150 flanged''',

'HC-CP4L-1500': '''HYDROCORE CP-4L/1500 CENTRIFUGAL PROCESS PUMP
Model / Part: HC-CP4L-1500
Wetted material: Duplex 2205
Max discharge pressure: 2.5 MPa
Operating temperature: 5 C ... 95 C
Dry weight: 82 kg | Rated flow: 45 m3/h''',

'VG-PR24': '''VOLTGUARD PR-24 PRESSURE TRANSMITTER
Part: VG-PR24 | Housing: 316 stainless steel
Proof pressure: 6.0 MPa
Media temperature: -40 to 125 C
Weight: 0.45 kg''',

'TF-HX-M12-1290': '''TITANFAST M12 HEX CAP SCREW - GRADE 12.9
Part number: TF-HX-M12-1290
Material: Alloy steel, zinc-plated
Proof load: 87 kN | Thread: M12 x 1.75
Mass per unit: 0.089 kg''',
}
print(f'{len(DATASHEETS)} datasheets')

## The schema we validate against

A Pydantic model is the contract. Optional fields are `None` when the datasheet does not
state them; the validators reject impossible values, which is what turns a schema from a
shape check into a correctness check.

In [ ]:
class Component(BaseModel):
    part_number: str
    material: str
    max_pressure_MPa: float | None = None       # None when not stated
    operating_temp_min_C: float | None = None
    operating_temp_max_C: float | None = None
    mass_kg: float | None = None

    @field_validator('max_pressure_MPa', 'mass_kg')
    @classmethod
    def positive(cls, v):
        if v is not None and v <= 0:
            raise ValueError('must be positive')
        return v

    @field_validator('max_pressure_MPa')
    @classmethod
    def plausible_pressure(cls, v):
        if v is not None and v > 1000:
            raise ValueError('pressure implausibly large for a component (MPa?)')
        return v

## The model call: one site, two backends

`call_model` is the only place that talks to an LLM. Offline it returns a canned,
provider-shaped response; with a key it calls the real provider. Everything downstream,
validation, repair, cost, scoring, is identical either way.

The stand-in is scripted to exercise the interesting cases: the improved prompt gets the
pump **wrong on its first attempt** (mass as a string) so we can watch the repair loop
fix it, and the naive prompt makes the two mistakes the notes warn about, an unconverted
`40 bar` and a hallucinated pressure for the bolt.

In [ ]:
# Canned responses: RESPONSES[mode][part] is a list of attempts. The first
# element is the model's first reply; later elements are what it returns after
# being asked to repair. Most parts are right the first time.
def _j(**kw):
    return json.dumps(kw)

GOLD_JSON = {
  'FS-BV200-SS': _j(part_number='FS-BV200-SS', material='SS316L stainless steel',
                    max_pressure_MPa=4.0, operating_temp_min_C=-20, operating_temp_max_C=180, mass_kg=3.4),
  'HC-CP4L-1500': _j(part_number='HC-CP4L-1500', material='Duplex 2205',
                     max_pressure_MPa=2.5, operating_temp_min_C=5, operating_temp_max_C=95, mass_kg=82.0),
  'VG-PR24': _j(part_number='VG-PR24', material='316 stainless steel',
                max_pressure_MPa=6.0, operating_temp_min_C=-40, operating_temp_max_C=125, mass_kg=0.45),
  'TF-HX-M12-1290': _j(part_number='TF-HX-M12-1290', material='Alloy steel, zinc-plated',
                       max_pressure_MPa=None, operating_temp_min_C=None, operating_temp_max_C=None, mass_kg=0.089),
}

# improved prompt: all correct, but the pump's first reply has mass as a string
# (a type error Pydantic will catch) and is repaired on the second attempt.
_pump_broken = _j(part_number='HC-CP4L-1500', material='Duplex 2205', max_pressure_MPa=2.5,
                  operating_temp_min_C=5, operating_temp_max_C=95, mass_kg='82 kg')
RESPONSES = {
  'improved': {k: [v] for k, v in GOLD_JSON.items()},
  'naive': {
     # forgot bar -> MPa (40 bar is 4.0 MPa)
     'FS-BV200-SS': [_j(part_number='FS-BV200-SS', material='SS316L stainless steel',
                        max_pressure_MPa=40.0, operating_temp_min_C=-20, operating_temp_max_C=180, mass_kg=3.4)],
     'HC-CP4L-1500': [GOLD_JSON['HC-CP4L-1500']],
     'VG-PR24': [GOLD_JSON['VG-PR24']],
     # invented a pressure and a temperature range for a bolt that has neither
     'TF-HX-M12-1290': [_j(part_number='TF-HX-M12-1290', material='Alloy steel, zinc-plated',
                           max_pressure_MPa=16.0, operating_temp_min_C=-20, operating_temp_max_C=150, mass_kg=0.089)],
  },
}
RESPONSES['improved']['HC-CP4L-1500'] = [_pump_broken, GOLD_JSON['HC-CP4L-1500']]

In [ ]:
def _est_tokens(text):
    return max(1, len(text) // 4)     # a rough token estimate for offline cost

def call_model(system, user, part_id, mode, attempt):
    '''Return (json_text, usage). Offline: canned. With a key: real provider.'''
    if HAVE_KEY and os.environ.get('ANTHROPIC_API_KEY'):
        import anthropic                        # only imported on the live path
        client = anthropic.Anthropic()
        resp = client.messages.create(model=MODEL, max_tokens=512, temperature=0,
            system=system, messages=[{'role': 'user', 'content': user}])
        return resp.content[0].text, {'input_tokens': resp.usage.input_tokens,
                                      'output_tokens': resp.usage.output_tokens}
    # offline stand-in
    attempts = RESPONSES[mode][part_id]
    text = attempts[min(attempt, len(attempts) - 1)]
    usage = {'input_tokens': _est_tokens(system + user), 'output_tokens': _est_tokens(text)}
    return text, usage

def cost_usd(usage):
    return (usage['input_tokens'] / 1e6 * PRICE_IN_PER_MTOK
            + usage['output_tokens'] / 1e6 * PRICE_OUT_PER_MTOK)

## The extract / validate / repair loop

The whole point: a validation error is fed back to the model as a repair request, not
raised. We cap the attempts and, if it never validates, flag the document instead of
writing a bad record.

In [ ]:
SYSTEM = {
  'naive': 'Extract the component fields as JSON.',
  'improved': ('You extract component data from datasheets into JSON matching the schema. '
     'Convert all pressures to MPa (1 MPa = 10 bar). If a field is not stated in the text, '
     'return null; never guess. A bolt proof load is not a pressure. '
     'Example: "Working pressure: 40 bar" -> max_pressure_MPa: 4.0.'),
}

def extract(part_id, text, mode, max_repairs=2, log=False):
    system, user = SYSTEM[mode], f'Datasheet:\n{text}'
    total = {'input_tokens': 0, 'output_tokens': 0}
    for attempt in range(max_repairs + 1):
        raw, usage = call_model(system, user, part_id, mode, attempt)
        for k in total: total[k] += usage[k]
        try:
            rec = Component.model_validate_json(raw)
            if log and attempt: print(f'    repaired on attempt {attempt + 1}')
            return rec, total, True
        except ValidationError as e:
            msg = e.errors()[0]
            if log: print(f'    attempt {attempt + 1} invalid: {msg["loc"]} {msg["msg"]}')
            user = (f'Datasheet:\n{text}\n\nYour previous JSON was {raw}\n'
                    f'It failed validation: {msg["loc"]}: {msg["msg"]}. Return corrected JSON.')
    return None, total, False

## Run it: a clean extraction, a repair, and a null

Watch three things: the pump is **repaired** after its first reply fails validation, the
bolt returns **null** pressure instead of a hallucinated number, and every call reports
its token usage and cost.

In [ ]:
table = []
for pid, text in DATASHEETS.items():
    print(f'{pid}:')
    rec, usage, ok = extract(pid, text, 'improved', log=True)
    print(f'    -> {rec.material}, {rec.max_pressure_MPa} MPa, mass {rec.mass_kg} kg '
          f'| {usage["input_tokens"]}+{usage["output_tokens"]} tok, {cost_usd(usage)*100:.3f} cents')
    table.append(rec)
print(f'\nextracted {len(table)} validated records')

In [ ]:
print('the bolt has no pressure rating; the extractor returned:')
bolt = [r for r in table if r.part_number == 'TF-HX-M12-1290'][0]
print(f'  max_pressure_MPa = {bolt.max_pressure_MPa!r}  (null, not a hallucinated number)')
valve = [r for r in table if r.part_number == 'FS-BV200-SS'][0]
print(f'  valve: 40 bar in the datasheet -> {valve.max_pressure_MPa} MPa in the record')

## Prompt evaluation on a gold set

The only way to know a prompt is better is to score it. We hand-label the correct answer
for each datasheet, then measure field-level accuracy for the naive prompt and the improved
one, and put the accuracy delta next to the cost delta.

In [ ]:
GOLD = {pid: Component.model_validate_json(js) for pid, js in GOLD_JSON.items()}
FIELDS = ['material', 'max_pressure_MPa', 'operating_temp_min_C', 'operating_temp_max_C', 'mass_kg']

def score(mode):
    right = total_fields = 0
    cost = 0.0
    for pid, text in DATASHEETS.items():
        rec, usage, ok = extract(pid, text, mode)
        cost += cost_usd(usage)
        for f in FIELDS:
            total_fields += 1
            if ok and getattr(rec, f) == getattr(GOLD[pid], f):
                right += 1
    return right / total_fields, cost

for mode in ('naive', 'improved'):
    acc, cost = score(mode)
    print(f'{mode:9s}  field accuracy {acc:5.1%}   cost {cost*100:.3f} cents for {len(DATASHEETS)} datasheets')

The improved prompt fixes the unconverted `40 bar` and the invented bolt pressure, so its
field accuracy is higher. It also costs more, because its instructions and example add
input tokens on every call. That is the real decision: **accuracy per dollar**, not
accuracy alone. With a real key, these numbers become your own model's.

---

## Takeaway

A reliable extractor is a loop, not a single call: constrain the output to a schema,
validate it with Pydantic, and repair on failure. Read token usage off every response so
cost is a number you design around, and score prompts on a small gold set so a change is
measured rather than guessed. A schema guarantees shape; the units, the null, and the gold
set are what guarantee the record is worth writing down. This is exactly assignment A8, at
the scale of four datasheets instead of a few hundred.